In [1]:
import langchain, langchain_core
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
load_dotenv()

/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [3]:
llm.invoke('Hi').content

'Hello! How can I assist you today?'

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [5]:
# prompt = 
prompt = ChatPromptTemplate.from_template(
    'Reply in one sentence: {question}'
)

parser = StrOutputParser()
chain = prompt | llm | parser

# https://python.langchain.com/docs/concepts/lcel/

result = chain.invoke('What is LLM')

In [6]:
classify_prompt = ChatPromptTemplate.from_template(
    'Classify this email in ONE word (Billing/Technical/Spam/Other):\n'
    'Subject: {subject}\nBody: {body}'
)

# chain
clasify_chain = classify_prompt | llm | parser

In [7]:
emails = [
    {'subject': 'Payment failed',   'body': 'Charged twice this month.'},
    {'subject': 'App crash',        'body': 'Crashes on Settings page.'},
    {'subject': 'Add Slack?',       'body': 'Want Slack notifications.'},
    {'subject': 'WIN FREE IPHONE',  'body': 'Click here NOW!!!'},
    {'subject': 'Invoice question', 'body': 'Can I get annual invoice?'},
]

In [8]:
emails[0]

{'subject': 'Payment failed', 'body': 'Charged twice this month.'}

In [9]:
# invoke / batch /stream
# compare for loop vs batch : time ??

import time
start = time.time()
loop_result = []
for email in emails:
    loop_result.append(clasify_chain.invoke(email))

loop_time = time.time()-start
print(loop_time)

3.908372163772583


In [10]:
loop_result

['Billing', 'Technical', 'Other', 'Spam', 'Billing']

In [11]:
batch_results = clasify_chain.batch(emails,
                                    config={'max_concurrency':3})

In [12]:
batch_results

['Billing', 'Technical', 'Other', 'Spam', 'Billing']

In [13]:
clasify_chain.with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True
)

RunnableRetry(bound=ChatPromptTemplate(input_variables=['body', 'subject'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['body', 'subject'], input_types={}, partial_variables={}, template='Classify this email in ONE word (Billing/Technical/Spam/Other):\nSubject: {subject}\nBody: {body}'), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice

In [14]:
clasify_chain.batch(emails)

['Billing', 'Technical', 'Other', 'Spam', 'Billing']

In [15]:
from langchain_core.runnables import RunnableLambda

In [16]:
# prompt
# prompt spam

# lcel

SPAM_KEYWORDS = ["lottery", "winner", "click here", "free money", "urgent", "congratulations", "prize"]

def validate_email(email):
    print("before parsing email: ",email['question'])
    body = email.get("question","").lower()
    print("after parsing email: ",email['question'])
    found_spam = [word for word in SPAM_KEYWORDS if word in body]
    print("found_spam",found_spam)
    if found_spam:
        raise ValueError("Spam word found:",found_spam)
    return email


validate_email({"question": "Congratulations! You are a lottery winner"})

before parsing email:  Congratulations! You are a lottery winner
after parsing email:  Congratulations! You are a lottery winner
found_spam ['lottery', 'winner', 'congratulations']


ValueError: ('Spam word found:', ['lottery', 'winner', 'congratulations'])

In [ ]:
simple_template = ChatPromptTemplate.from_template(
    'Reply in ONE sentence: {question}'
)

In [ ]:
chain = RunnableLambda(validate_email) | simple_template | llm | parser

In [ ]:
def run_chain(email_body):
    try:
        result = chain.invoke({"question": email_body})
        print('llm response:',result)
    except ValueError as e:
        print("Blocked",e)

In [ ]:
# test case 1: 
run_chain("Can you help me fix my Python code?")

before parsing email:  Can you help me fix my Python code?
after parsing email:  Can you help me fix my Python code?
found_spam []
llm response: Absolutely! Please share your Python code, and I'll assist you in fixing it.


In [ ]:
# test case 2:
run_chain("Congratulations! You are a lottery winner, click here to claim your free money!".lower())


before parsing email:  congratulations! you are a lottery winner, click here to claim your free money!
after parsing email:  congratulations! you are a lottery winner, click here to claim your free money!
found_spam ['lottery', 'winner', 'click here', 'free money', 'congratulations']
Blocked ('Spam word found:', ['lottery', 'winner', 'click here', 'free money', 'congratulations'])


In [ ]:
# parsing --> string

'Hello! How can I assist you today?'

In [ ]:
# pydanticparser ==Langchain ===> using pydantic library ==> validate your input and output

In [ ]:
# plain python way of calling model
outputs_from_llm = [
    'CATEGORY: Billing\nURGENCY: High',        # normal
    'CATEGORY:  Billing\nURGENCY: High',       # extra space!
    'Category: Billing\nUrgency: High',         # different case!
    'CATEGORY: Billing (payment issue)\nURGENCY: High',  # extra text!
    'I think this is CATEGORY: Billing\nURGENCY: High',  # explanation first!
]


# here you need to write the logic by yourself ?? this is like time consuming ??
# pydanticparser


In [18]:
from pydantic import BaseModel, Field

from typing import Literal, List

In [ ]:
# output format ??

class EmailClassification(BaseModel):
    category : Literal['Billing','technical','Feature request','Other'] = Field(description='the category of the support email')
    urgency: Literal['High', 'Medium', 'Low'] = Field(
        description='Urgency based on business impact'
    )

    confidence: int = Field(
        description='Confidence score from 1 to 10',
        ge=1,   # ge = greater than or equal (Pydantic v2!)
        le=10   # le = less than or equal
    )

    reasoning: str = Field(
        description='One sentence explanation of the classification'
    )





   



In [41]:
EmailClassification_input(
    subject=2,
    body='13123123'
)

ValidationError: 1 validation error for EmailClassification_input
subject
  Input should be a valid string [type=string_type, input_value=2, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

In [23]:
# EmailClassification(
#     category='fsfdsfds',
#     urgency='High',
#     confidence=9,
#     reasoning='Hi this is good'
# )

In [24]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=EmailClassification)

In [27]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"category": {"description": "the category of the support email", "enum": ["Billing", "technical", "Feature request", "Other"], "title": "Category", "type": "string"}, "urgency": {"description": "Urgency based on business impact", "enum": ["High", "Medium", "Low"], "title": "Urgency", "type": "string"}, "confidence": {"description": "Confidence score from 1 to 10", "maximum": 10, "minimum": 1, "title": "Confidence", "type": "integer"}, "reasoning": {"description": "One sentence explanation of the classification", "title"

In [28]:
cot_prompt = ChatPromptTemplate.from_messages([
    ('system', '''You are an expert support email classifier.

Classification Rules:
- Login broken after payment → Billing (NOT Technical)
- App crashes → Technical
- Pricing complaints + evaluating alternatives → Churn Risk
- Feature requests → Feature Request

Think step by step before classifying.

{format_instruction}'''),
     ('human', 'Subject: {subject}\nBody: {body}')
]).partial(format_instruction= parser.get_format_instructions())

In [ ]:
cot_chain = cot_prompt | llm | parser # entire chain 

In [30]:
email = {
    'subject': "Can't login — paid for annual plan last week",
    'body': 'Our entire team cannot login. We paid for the annual plan '
            'last week. We have a board demo in 3 hours!'
}


In [38]:
result = cot_chain.invoke(email)

In [35]:
result

EmailClassification(category='Billing', urgency='High', confidence=9, reasoning='The login issue is directly linked to a payment that was made recently, indicating a billing-related problem rather than a technical one.')

In [37]:
result.confidence

9

In [ ]:
# chain = simple_template | validate_email | llm | parser



In [ ]:
chain.invoke({'question':"Congratulations! You are a lottery winner, click here to claim your free money!"})

"It seems you're referencing a method from the IPython kernel that handles user input, but it's not being called properly."

In [ ]:
email = {
    'questions':'Hi I am ok'
}
body = email.get("questions","").lower()

In [ ]:
body

'hi i am ok'